<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_15_git_github_system/note_lesson_15_git_github_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 15 — Git + GitHub: двоє в одному репозиторії

Команда кафе збирає звіт каси з шести задач ([`team_project/`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/tree/main/module_1/lessons/lesson_15_git_github_system/team_project)). Перш ніж працювати вшістьох на GitHub, пройдемо весь шлях самі в тимчасовому репозиторії `team-demo/`. По черзі граємо трьох людей:

- **Ірина** — лідерка, створює репозиторій і зливає Pull Request-и;
- **Оксана** — задача 1, `rules.py`;
- **Тарас** — задача 4, `stats.py`.

Клітинки `%%bash` виконують команди термінала: у Colab — одразу, локально — у Linux/macOS або в Jupyter з Git Bash на Windows. Кожна клітинка `%%bash` — окремий термінал, тому починається з `cd team-demo`. Виконуй **зверху вниз**; перед **Прогнозом** спершу скажи, що буде.

Теорія — у книзі: [Урок 15. Git + GitHub: командний проєкт](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_15/).

## 🔁 Пригадай (без підглядання)

1. Які команди ти виконуєш між «код домашньої написано» і «PR відкрито»?
2. Що показує `git status`?
3. Чим `origin` відрізняється від `upstream` у fork курсу?

<details>
<summary>Відповіді</summary>

1. `git switch -c homework-NN`, `git add`, `git commit -m "..."`, `git push origin homework-NN`, **Compare & pull request**.
2. Змінені, підготовлені до коміту й невідстежувані файли, поточну гілку.
3. `origin` — твій fork (туди `push`), `upstream` — репозиторій викладача (звідти нові уроки).

</details>

## 0. Шаблон проєкту

Беремо `team_project/` з репозиторію курсу (у Colab клітинка спершу клонує курс) і копіюємо в `team-demo/` без теки `solution/`.

In [ ]:
import shutil
import subprocess
from pathlib import Path

COURSE = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"
source = Path("team_project")                     # ноутбук відкрито з теки уроку
if not source.exists():                           # Colab: клонуємо курс один раз
    if not Path("course").exists():
        subprocess.run(["git", "clone", "--depth", "1", COURSE, "course"], check=True)
    source = Path("course/module_1/lessons/lesson_15_git_github_system/team_project")

shutil.rmtree("team-demo", ignore_errors=True)
shutil.copytree(source, "team-demo", ignore=shutil.ignore_patterns("solution", "__pycache__"))
print(sorted(path.name for path in Path("team-demo").iterdir()))

## 1. Ірина створює репозиторій

`git config user.name` без `--global` задає ім'я лише для цього репозиторію — твої глобальні налаштування не зміняться.

In [ ]:
%%bash
cd team-demo
git init -q -b main
git config user.name "Ірина"
git config user.email "iryna@example.com"
git status --short

**Прогноз:** що покаже `git status --short` після `git add .`? А після `git commit`?

In [ ]:
%%bash
cd team-demo
git add .
git status --short | head -4
echo "..."
git commit -q -m "Шаблон проєкту: контракти, тести, заглушки"
echo "--- після коміту:"
git status --short
git log --oneline

<details>
<summary>Відповідь</summary>

Після `git add .` — `A` (added) у першій колонці: файли в staging. Після коміту `git status --short` порожній: робоча тека збігається з останнім комітом.

</details>

In [ ]:
%%bash
cd team-demo
python3 check.py

## 2. Оксана: задача 1 у своїй гілці

Гілка — лише вказівник на коміт, файли не копіюються.

In [ ]:
%%bash
cd team-demo
git config user.name "Оксана"
git config user.email "oksana@example.com"
git switch -c task-1-rules
git branch

### 🛠 Вправа 1. Напиши `rules.py` за контрактом

Контракт — у докстрінгах. `DAYS` уже є в `models.py`. Клітинка записує файл у `team-demo/`, наступна запускає тести задачі.

In [ ]:
%%writefile team-demo/rules.py
"""Задача 1. Правила кафе: день тижня і прийом їжі."""
from models import DAYS


def meal_type(hour):
    """Прийом їжі за годиною: 11–15 → "обід", 17–23 → "вечеря", решта → "інше".

    >>> meal_type(12), meal_type(16), meal_type(18)
    ('обід', 'інше', 'вечеря')
    """
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if 11 <= hour <= 15:
        return "обід"
    if 17 <= hour <= 23:
        return "вечеря"
    return "інше"
    # END SOLUTION


def day_name(timestamp):
    """Короткий день тижня з DAYS: datetime(2024, 7, 19, 18, 30) → "пт"."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return DAYS[timestamp.weekday()]
    # END SOLUTION

In [ ]:
%%bash
cd team-demo
python3 check.py 1

Тести `OK` → дописуємо себе в `TEAM.md` і комітимо **лише свої** файли.

In [ ]:
%%bash
cd team-demo
echo "| 1 | rules.py | Оксана |" >> TEAM.md
git add rules.py TEAM.md
git status --short
git commit -q -m "Задача 1: meal_type і day_name"
git log --oneline

## 3. Тарас: задача 4 — паралельно з Оксаною

Тарас починає від `main`, де **ще немає** коду Оксани. Його `stats.py` викликає `day_name` і `meal_type` з `rules.py` — а там у `main` заглушки.

**Прогноз:** чи зможе Тарас перевірити свою задачу?

In [ ]:
%%bash
cd team-demo
git config user.name "Тарас"
git config user.email "taras@example.com"
git switch main
git switch -c task-4-stats
grep -c NotImplementedError rules.py

### 🛠 Вправа 2. Напиши `stats.py` за контрактом

Спирайся лише на **контракт** `day_name` і `meal_type`, не на їхній код.

In [ ]:
%%writefile team-demo/stats.py
"""Задача 4. Підрахунки за днями і прийомами їжі."""
from rules import day_name, meal_type


def revenue_by_day(orders):
    """Виторг за днями тижня: {"пт": 860.0, ...}. Лише дні, у які були чеки."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    revenue = {}
    for order in orders:
        day = day_name(order.timestamp)
        revenue[day] = revenue.get(day, 0) + order.total_bill
    return revenue
    # END SOLUTION


def best_day(revenue):
    """День з найбільшим виторгом; для порожнього словника — None."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    best = None
    for day, amount in revenue.items():
        if best is None or amount > revenue[best]:
            best = day
    return best
    # END SOLUTION


def count_by_meal(orders):
    """Кількість чеків за прийомом їжі: {"вечеря": 3, "обід": 1}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    counts = {}
    for order in orders:
        meal = meal_type(order.timestamp.hour)
        counts[meal] = counts.get(meal, 0) + 1
    return counts
    # END SOLUTION

In [ ]:
%%bash
cd team-demo
python3 check.py 4

<details>
<summary>Відповідь</summary>

Так: тести задачі 4 підставляють заглушки `fake_day_name` і `fake_meal_type` (подивись `tests/test_task4.py`), тож задача 4 перевіряється, хоча в гілці Тараса `rules.py` ще не написано. Це і є робота за контрактом.

</details>

In [ ]:
%%bash
cd team-demo
echo "| 4 | stats.py | Тарас |" >> TEAM.md
git add stats.py TEAM.md
git commit -q -m "Задача 4: виторг за днями, найкращий день, прийоми їжі"
git log --oneline --graph --all

## 4. Ірина зливає PR #1

На GitHub це кнопка **Merge pull request**; локально — `git merge --no-ff`: коміт злиття з двома батьками.

In [ ]:
%%bash
cd team-demo
git config user.name "Ірина"
git config user.email "iryna@example.com"
git switch -q main
git merge --no-ff task-1-rules -m "Merge pull request #1 from task-1-rules"

## 5. Конфлікт

Тарас підтягує свіжий `main` у свою гілку, щоб його PR злився чисто.

**Прогноз:** які файли зіллються самі, а який — ні? Чому?

In [ ]:
%%bash
cd team-demo
git config user.name "Тарас"
git config user.email "taras@example.com"
git switch -q task-4-stats
git merge main
git status --short
echo "--- TEAM.md:"
cat TEAM.md

<details>
<summary>Відповідь</summary>

`rules.py` і `stats.py` — кожен змінювала лише одна гілка — зливаються самі. `TEAM.md` конфліктує: обидві гілки дописали рядок **в одне місце**. `UU` — файл з конфліктом.

</details>

### 🛠 Вправа 3. Розв'яжи конфлікт

Можна відкрити `team-demo/TEAM.md` у редакторі (у Colab — подвійний клік на панелі 📁) або виправити кодом нижче. Потрібні **обидва** рядки: спершу задача 1, потім 4. Усі три маркери — `<<<<<<<`, `=======`, `>>>>>>>` — прибрати.

In [ ]:
team = Path("team-demo/TEAM.md")
text = team.read_text(encoding="utf-8")
# YOUR CODE HERE
# BEGIN SOLUTION
lines = [line for line in text.splitlines()
         if not line.startswith(("<<<<<<<", "=======", ">>>>>>>"))]
rows = sorted(line for line in lines if line.startswith("| ") and line[2].isdigit())
text = "\n".join([line for line in lines if line not in rows] + rows) + "\n"
team.write_text(text, encoding="utf-8")
# END SOLUTION

text = team.read_text(encoding="utf-8")
print(text)
assert not any(marker in text for marker in ("<<<<<<<", "=======", ">>>>>>>")), "лишилися маркери"
assert text.index("| 1 | rules.py") < text.index("| 4 | stats.py"), "спершу задача 1, потім 4"
print("✅ Вправа 3 пройдена")

Кроки 3 і 4: `git add` позначає конфлікт розв'язаним, `git commit` завершує злиття. Потім Ірина зливає PR #2.

In [ ]:
%%bash
cd team-demo
git add TEAM.md
git commit -q --no-edit
git config user.name "Ірина"
git config user.email "iryna@example.com"
git switch -q main
git merge -q --no-ff task-4-stats -m "Merge pull request #2 from task-4-stats"
git log --oneline --graph
python3 check.py

## 6. `.gitignore`

Запуски створили `__pycache__/`, а `main.py` створює `report.json`. **Прогноз:** що покаже `git status --short`?

In [ ]:
%%bash
cd team-demo
echo "{}" > report.json
ls
echo "--- git status --short:"
git status --short
git check-ignore -v report.json

<details>
<summary>Відповідь</summary>

Нічого: `__pycache__/` і `report.json` є в `.gitignore`, Git їх не показує. `git check-ignore -v` пояснює, яке правило спрацювало.

</details>

## 7. Перевірка репозиторію

In [ ]:
def git(*args):
    return subprocess.run(["git", *args], cwd="team-demo", capture_output=True, text=True).stdout


commits = git("log", "--oneline").splitlines()
merged = git("branch", "--merged", "main")
print(len(commits), "комітів;", "гілки злиті:", " ".join(merged.split()))
assert len(commits) >= 6, "шаблон, 2 задачі, злиття гілки Тараса з main, 2 PR"
assert "task-1-rules" in merged and "task-4-stats" in merged
assert "report.json" not in git("ls-files")
team = Path("team-demo/TEAM.md").read_text(encoding="utf-8")
assert "<<<<<<<" not in team and "Оксана" in team and "Тарас" in team
print("✅ Репозиторій у порядку")

### 🛠 Вправа 4. Файл, який не мав потрапити в коміт

Хтось «спростив» `.gitignore`, і `report.json` потрапив у коміт. Клітинка нижче відтворює це в гілці `fix-ignore`.

In [ ]:
%%bash
cd team-demo
git switch -q -c fix-ignore
grep -v "^report.json$" .gitignore > .gitignore.new && mv .gitignore.new .gitignore
git add .
git commit -q -m "Спроба: report.json потрапив у репозиторій"
git ls-files | grep report

Виправ у цій самій гілці: поверни `report.json` у `.gitignore`, прибери файл **з репозиторію, але не з диска** (`git rm --cached`), закоміть.

In [ ]:
%%bash
cd team-demo
# YOUR CODE HERE
# BEGIN SOLUTION
echo "report.json" >> .gitignore
git rm -q --cached report.json
git add .gitignore
git commit -q -m "Не зберігати report.json: його генерує main.py"
# END SOLUTION
git status --short

In [ ]:
assert "report.json" not in git("ls-files"), "report.json досі в репозиторії"
assert Path("team-demo/report.json").exists(), "файл на диску має лишитися"
assert git("status", "--short") == "", "робоча тека має бути чистою"
print("✅ Вправа 4 пройдена")

## 8. Портфоліо-мінімум

Проєкт, який не соромно показати: README пояснює, що це і як запустити; `.gitignore` не пускає сміття й секрети; історія комітів читається. Два маленькі чеклисти:

In [ ]:
def check_readme(text):
    """Мінімальний чеклист README: заголовок, опис, інструкція запуску."""
    lowered = text.lower()
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() and not p.strip().startswith("#")]
    return {
        "має заголовок (# Назва)": text.strip().startswith("#"),
        "має опис проєкту": len(paragraphs) > 0,
        "пояснює, як запустити": "```" in text or "запуст" in lowered,
    }


def check_gitignore(text, required=("__pycache__/", ".venv", "*.pyc", ".env")):
    """Мінімальний чеклист .gitignore для Python-проєкту."""
    lines = [line.strip() for line in text.splitlines()]
    return {pattern: any(pattern in line for line in lines) for pattern in required}


readme = check_readme(Path("team-demo/README.md").read_text(encoding="utf-8"))
gitignore = check_gitignore(Path("team-demo/.gitignore").read_text(encoding="utf-8"))
print(readme)
print(gitignore)
assert all(readme.values()) and all(gitignore.values())
assert not all(check_readme("TODO: написати опис").values())

## 9. Тепер — командою на GitHub

1. Лідер створює публічний репозиторій ([інструкція](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/create_repository/)), копіює туди вміст `team_project/` **без** `solution/`, робить перший коміт і `git push`.
2. **Settings → Collaborators → Add people** — додає команду.
3. Кожен: `git switch main` → `git pull` → `git switch -c task-N-…` → код → `python check.py N` → рядок у `TEAM.md` → `git add` своїх файлів → `git commit` → `git push -u origin task-N-…` → PR.
4. Рев'ю іншої людини → Merge. Конфлікт у `TEAM.md` розв'язує автор PR (`git merge main` у своїй гілці).
5. Готово, коли в `main` `python check.py all` друкує `✅ Проєкт працює!`.

## ✅ Самоперевірка

1. Що робить `git add` і навіщо він окремо від `git commit`?
2. Чи копіює `git switch -c` файли?
3. Чому `rules.py` і `stats.py` злилися самі, а `TEAM.md` — ні?
4. Git злив усі PR без конфліктів. Чи означає це, що проєкт працює?
5. Як Тарас перевірив задачу 4, коли задача 1 ще не була готова?

<details>
<summary>Відповіді</summary>

1. Кладе стан файлу в staging — набір наступного коміту; можна закомітити лише частину змін.
2. Ні: гілка — вказівник на коміт.
3. Кожен з цих файлів змінювала одна гілка; `TEAM.md` — обидві, в одному місці.
4. Ні: Git порівнює текст, а не сенс. Перевіряє `python check.py all`.
5. Тести підставили заглушки замість `day_name` і `meal_type`; задача 4 спирається лише на їхній контракт.

</details>

### Шпаргалка

```bash
git init -b main                 # новий репозиторій
git status --short               # ?? — не відстежується, M — змінено, A — додано, UU — конфлікт
git add файл                     # у staging
git commit -m "Задача 3: що і навіщо"
git log --oneline --graph --all  # історія
git switch -c task-3-loading     # нова гілка
git switch main && git pull      # свіжий main
git push -u origin task-3-loading
git merge main                   # підтягнути main у свою гілку
git merge --abort                # скасувати злиття з конфліктом
git rm --cached report.json      # прибрати з репозиторію, лишити на диску
git check-ignore -v файл         # чому файл ігнорується
```

## Далі

**Урок 16 — Практикум 3. Хеш-структури.** Капстоун уроку 17 ти вестимеш у власному репозиторії так само: гілки, коміти, `.gitignore`, README.